<a href="https://colab.research.google.com/github/alldi1212/Metode-AI/blob/main/metode_Ai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q kaggle

import os
import shutil
import random
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from google.colab import files
from tensorflow.keras import layers, models
from tensorflow.keras.applications import VGG19
from tensorflow.keras.applications.vgg19 import preprocess_input
from tensorflow.keras.preprocessing import image_dataset_from_directory

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

TensorFlow: 2.21.0
GPU: []


In [ ]:
# Unggah file kaggle.json Anda ketika diminta
# files.upload()

# !mkdir -p ~/.kaggle
# !cp kaggle.json ~/.kaggle/
# !chmod 600 ~/.kaggle/kaggle.json

# Contoh perintah download dataset (ganti dengan link dataset Anda)
# !kaggle datasets download -d <username>/<nama-dataset>
# !unzip -q <nama-dataset>.zip -d dataset/

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# Sesuaikan path direktori dengan folder dataset Anda
train_dir = "dataset/train"
val_dir = "dataset/validation"

try:
  train_dataset = image_dataset_from_directory(
      train_dir,
      image_size=IMG_SIZE,
      batch_size=BATCH_SIZE,
      label_mode="categorical",
      shuffle=True,
  )

  val_dataset = image_dataset_from_directory(
      val_dir,
      image_size=IMG_SIZE,
      batch_size=BATCH_SIZE,
      label_mode="categorical",
      shuffle=False,
  )

  class_names = train_dataset.class_names
  NUM_CLASSES = len(class_names)
  print("Kelas terdeteksi:", class_names)
except Exception as e:
  print(
      "Catatan: Direktori dataset belum ditemukan. Sesuaikan path folder"
      " Anda."
  )
  NUM_CLASSES = 2  # Nilai default sementara

Found 10 files belonging to 2 classes.
Found 10 files belonging to 2 classes.
Kelas terdeteksi: ['kelas_a', 'kelas_b']


In [ ]:
# Memuat base model VGG19 tanpa bagian fully connected atasnya
base_model = VGG19(
    weights="imagenet", include_top=False, input_shape=(224, 224, 3)
)

# Membekukan layer base model agar bobot dasarnya tidak berubah
base_model.trainable = False

# Menyusun model kustom
model = models.Sequential(
    [
        layers.Lambda(preprocess_input, input_shape=(224, 224, 3)),
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation="relu"),
        layers.Dropout(0.5),
        layers.Dense(NUM_CLASSES, activation="softmax"),
    ]
)

model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lambda_3 (Lambda)               │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ vgg19 (Functional)              │ (None, 7, 7, 512)      │    20,024,384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_3      │ (None, 512)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 2)              │           514 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20,156,226 (76.89 MB)

 Trainable params: 131,842 (515.01 KB)

 Non-trainable params: 20,024,384 (76.39 MB)

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

In [ ]:
import os
import numpy as np
from PIL import Image

# Membuat folder dummy untuk latihan
os.makedirs("dataset/train/kelas_a", exist_ok=True)
os.makedirs("dataset/train/kelas_b", exist_ok=True)
os.makedirs("dataset/validation/kelas_a", exist_ok=True)
os.makedirs("dataset/validation/kelas_b", exist_ok=True)

# Membuat gambar dummy warna acak ukuran 224x224
for folder in [
    "dataset/train/kelas_a",
    "dataset/train/kelas_b",
    "dataset/validation/kelas_a",
    "dataset/validation/kelas_b",
]:
  for i in range(5):
    img_array = np.random.rand(224, 224, 3) * 255
    img = Image.fromarray(img_array.astype("uint8"))
    img.save(os.path.join(folder, f"img_{i}.jpg"))

print(
    "Folder dataset dummy berhasil dibuat! Silakan jalankan ulang kode"
    " pemuatan dataset dan training Anda."
)

Folder dataset dummy berhasil dibuat! Silakan jalankan ulang kode pemuatan dataset dan training Anda.


In [ ]:
def plot_history(history):
  acc = history.history["accuracy"]
  val_acc = history.history["val_accuracy"]
  loss = history.history["loss"]
  val_loss = history.history["val_loss"]

  epochs_range = range(len(acc))

  plt.figure(figsize=(12, 4))
  plt.subplot(1, 2, 1)
  plt.plot(epochs_range, acc, label="Training Accuracy")
  plt.plot(epochs_range, val_acc, label="Validation Accuracy")
  plt.legend(loc="lower right")
  plt.title("Training and Validation Accuracy")

  plt.subplot(1, 2, 2)
  plt.plot(epochs_range, loss, label="Training Loss")
  plt.plot(epochs_range, val_loss, label="Validation Loss")
  plt.legend(loc="upper right")
  plt.title("Training and Validation Loss")
  plt.show()


# Panggil fungsi jika training sudah dilakukan
# plot_history(history)



Pertanyaan analisis

Apa perbedaan training dari scratch dan transfer learning?
Mengapa pretrained model dapat belajar lebih cepat?
Apa fungsi freezing layer?
Apa perbedaan feature extraction dan fine-tuning?
Apakah fine-tuning selalu meningkatkan accuracy?
Bagaimana pengaruh jumlah data terhadap pemilihan strategi?


1. Apa perbedaan training dari scratch dan transfer learning?
Training dari Scratch (Dari Nol): Model dilatih menggunakan bobot awal yang diinisialisasi secara acak. Pendekatan ini membutuhkan dataset yang sangat besar dan waktu komputasi (GPU) yang sangat lama agar model dapat mengenali pola dari awal.

Transfer Learning: Menggunakan model yang sudah pernah dilatih sebelumnya (biasanya pada dataset raksasa seperti ImageNet) untuk menyelesaikan masalah baru. Kita hanya memanfaatkan "pengetahuan" yang sudah dimiliki model tersebut dan melatih ulang bagian lapisan atas (classifier) sesuai kebutuhan kita.

2. Mengapa pretrained model dapat belajar lebih cepat?
Pretrained model dapat belajar lebih cepat karena tidak perlu lagi belajar mengenali fitur-fitur dasar. Pada lapisan-lapisan awal (lower layers), model sudah belajar mengenali bentuk fundamental seperti garis, tepi, sudut, dan tekstur dari jutaan gambar sebelumnya. Ketika kita melatihnya dengan data baru, model tinggal fokus menyesuaikan pemahaman tersebut untuk mengenali objek spesifik di dataset kita, sehingga proses konvergensi (pencapaian akurasi tinggi) tercapai jauh lebih cepat.

3. Apa fungsi freezing layer?
Menjaga Pengetahuan Dasar: Membekukan layer (freezing) berarti mengunci bobot di dalam layer tersebut agar tidak berubah selama proses training. Ini mencegah hilangnya kemampuan dasar model (catastrophic forgetting).

Efisiensi Komputasi: Karena bobot tidak diubah, komputer tidak perlu menghitung gradien untuk layer yang dibekukan, sehingga proses backpropagation menjadi lebih cepat dan menghemat memori GPU.

4. Apa perbedaan feature extraction dan fine-tuning?
Feature Extraction: Seluruh lapisan base model dibekukan (frozen). Base model hanya bertindak sebagai ekstraktor fitur tetap, dan kita hanya melatih lapisan tambahan (dense layer baru) di atasnya.

Fine-Tuning: Setelah tahap feature extraction selesai atau sejak awal, kita membuka beberapa lapisan teratas dari base model (unfrozen) agar ikut dilatih bersama dengan lapisan baru menggunakan learning rate yang sangat kecil. Tujuannya agar model dapat mengadaptasi fitur spesifik dari dataset baru secara lebih mendalam.

5. Apakah fine-tuning selalu meningkatkan accuracy?
Tidak selalu. Fine-tuning memiliki beberapa risiko:

Overfitting: Jika dataset baru Anda terlalu kecil, proses fine-tuning pada banyak layer justru akan membuat model menghafal data latih alih-alih mempelajari pola umumnya.

Catastrophic Forgetting: Jika learning rate yang digunakan terlalu besar saat fine-tuning, bobot pretrained yang sudah bagus bisa rusak total, yang justru menurunkan akurasi model.

6. Bagaimana pengaruh jumlah data terhadap pemilihan strategi?
Jumlah data menjadi penentu utama dalam memilih strategi yang tepat:

Data Sangat Sedikit: Gunakan Feature Extraction (dengan frozen base model) untuk mencegah overfitting.

Data Sedang (Moderat): Gunakan feature extraction terlebih dahulu, lalu lakukan Fine-Tuning secara hati-hati pada beberapa layer terakhir dengan learning rate kecil.

Data Sangat Banyak (Melimpah): Anda memiliki kebebasan lebih besar; Anda bisa melakukan fine-tuning secara luas pada sebagian besar layer, atau bahkan melakukan training dari scratch jika domain datanya sangat unik (misalnya citra medis khusus atau satelit langka).

